In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer

from category_encoders import TargetEncoder

In [ ]:
df = pd.read_excel("../data/cleaned/df_cleaned.xlsx")

In [ ]:
df.head()

In [ ]:
X = df.drop(labels='COEMISSIONS', axis=1)
y = df["COEMISSIONS"]

In [ ]:
X.dtypes

In [ ]:
categorical_cols = ['TRANSMISSION', 'FUEL', 'MAKE', 'MODEL']
numeric_cols = ['ENGINE SIZE', 'CYLINDERS', 'FUEL CONSUMPTION']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

In [ ]:
# Column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("target_enc", TargetEncoder(smoothing=10), categorical_cols),
        ("num", RobustScaler(), numeric_cols)
    ]
)

# Full pipeline
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", ElasticNet(max_iter=10000))
])

# Hyperparameter grid
param_grid = {
    "model__alpha": np.logspace(-4, 1, 20),
    "model__l1_ratio": np.linspace(0.1, 0.9, 9)
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

grid.fit(X_train, y_train)

In [ ]:
print("Best parameters:", grid.best_params_)

In [ ]:
y_pred = grid.predict(X_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import math

mse = mean_squared_error(y_test, y_pred)
rmse = math.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

performance_matrix = pd.DataFrame({"Values":[mse,rmse,mae,r2]}, index=["MSE","RMSE","MAE","R2 Score"])
performance_matrix

In [ ]:
sns.jointplot(x=y_test, y=y_pred, kind='reg', line_kws={"color": "red"})
plt.xlabel("Actual Output")
plt.ylabel("Predicted Output")
plt.savefig("reports/actual_vs_predicted_output.png", dpi=300, bbox_inches="tight")
plt.show()

residuals = y_test - y_pred
# densityplot this residuals
sns.kdeplot(data=residuals,fill=True, color='g')
plt.savefig("reports/residual_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

# scatter plot with respect to prediction and residuals
plt.scatter(x=y_pred, y=residuals)
plt.xlabel("Predicted Output")
plt.ylabel("Residuals")
plt.savefig("reports/predicted_vs_residual.png", dpi=300, bbox_inches="tight")
plt.show()